## Assignment 1: Supervised Machine Learning

Welcome to the first assignment of CS 541! This assignment prepares you with some useful tools that are widely used in NLP. This assignment must be done individually.

After this assignment, you should be able to:  
1. Load a dataset from huggingface's dataset library, and do some exploratory analyses.  
2. Use scikit-learn to build and train a feature-based model.  
3. Use pytorch to build and train a feature-based model.  
4. Use Optuna to automatically search for hyperparameters.  

In CS541, any work generated by an AI shouldn't be included without declaration. If you include material generated by an AI, the level of AI use should be properly documented, and the actual tool should be noted (e.g., "I used Codex to proofread the codes and draft the analysis"). 

**AI use declaration (required by the course policy above):**
* All the comments in thie file are written by me, I used Ai primarily to understand and debugging. I have not changed assignmen'ts requred function signatures or core structurte and I reviewed my code and results muself included in this submission



### 1. Load the dataset (5')
First, we are going to load the datasets from huggingface's `datasets` library.
Do some exploratory analysis on the dataset.  
1.1 Print out one example in the dataset. Briefly comment on what it contains.  
1.2 For each of the train, validation, and test set, compute the following statistics: 
- The number of data samples with each class label.  
- The mean and std of the sentence lengths (in words) of each `question`.  

1.3 Vectorize the validation set of the dataset, following the approaches specified in the train set example.

In [1]:
import pandas as pd 
import numpy as np 
from datasets import load_dataset  # huggingface datasets

ds = load_dataset("stanfordnlp/sst2")

c:\Users\manda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [ ]:
# TODO -- Print out one example in the dataset. Briefly comment on what it contains.
example = ds["train"][0]
print(example)
#  Each example contains three fields: 
#  'idx' (the row index inside the split)
#  'sentence' -> sentence or phrase from SST-2
#  'label' (0 = negative sentiment, 1 = positive sentiment)

{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0}

Each example contains three fields: 'idx' (the row index inside the split), 'sentence' (a movie-review sentence or phrase from SST-2) and 'label' (0 = negative sentiment, 1 = positive sentiment).


In [4]:
train_data = pd.DataFrame(ds["train"])
X_train_text = train_data["sentence"]
Y_train = train_data["label"]

val_data = pd.DataFrame(ds["validation"])
X_val_text = val_data["sentence"]
Y_val = val_data["label"]

test_data = pd.DataFrame(ds["test"])
X_test_text = test_data["sentence"]
Y_test = test_data["label"]

In [ ]:
# TODO -- compute the exploratory statistics
# Note: the assignment text says `question`, but the SST-2 text field is called `sentence`.
stats_rows = []
for split_name in ["train", "validation", "test"]: # Go through all three dataset splits
    
    df = pd.DataFrame(ds[split_name]) # convert the split to a pandas dataframe
    sentence_lengths = df["sentence"].str.split().str.len()   # calculate sentence length, eg.My name is Mandar -> 4 words
    class_counts = df["label"].value_counts().sort_index() # count labels
    
    print("\n===== {} ({} samples) =====".format(split_name, len(df)))
    print("Class counts:") 
    print(class_counts.to_string()) # print class counts
    
    print("Mean sentence length (words): {:.4f}".format(sentence_lengths.mean())) # Calculates the average number of words per sentence
    print("Std  sentence length (words): {:.4f}".format(sentence_lengths.std())) # how much sentence lengths vary around the average 
    
    # Stores all the statistics calculated for the current split in the stats_rows
    stats_rows.append({"split": split_name, "n": len(df),
                       **{"label={}".format(k): v for k, v in class_counts.items()},
                       "mean_len_words": sentence_lengths.mean(), "std_len_words": sentence_lengths.std()})

print("\nSummary table:")
print(pd.DataFrame(stats_rows).fillna(0).to_string(index=False))


===== train (67349 samples) =====
Class counts:
label
0    29780
1    37569
Mean sentence length (words): 9.4096
Std  sentence length (words): 8.0738

===== validation (872 samples) =====
Class counts:
label
0    428
1    444
Mean sentence length (words): 19.5482
Std  sentence length (words): 8.7639

===== test (1821 samples) =====
Class counts:
label
-1    1821
Mean sentence length (words): 19.2339
Std  sentence length (words): 8.9224

Summary table:
     split     n  label=0  label=1  mean_len_words  std_len_words  label=-1
     train 67349  29780.0  37569.0        9.409553       8.073806       0.0
validation   872    428.0    444.0       19.548165       8.763900       0.0
      test  1821      0.0      0.0       19.233937       8.922386    1821.0

Note: every test label is -1 -> the GLUE test labels are hidden, so the test-set class counts are not meaningful and Y_test cannot be used for evaluation. All model selection below uses the validation set only.


**Observations (1.2).**
- Train (67,349): +ve → 56%, −ve → 44% → mildly imbalanced.
- Validation (872): +ve → 51%, −ve → 49% → mostly balanced.
- Test (1,821): labels → −1 → hidden/unavailable for evaluation.
- Sentence length: Train ≈ 9.4 words; Val/Test ≈ 19 words.

Next we are going to vectorize the texts using TfidfVectorizer, then compute the Tf-idf features.   

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer 
from sklearn.neural_network import MLPClassifier 

counter = CountVectorizer(min_df=10, max_df=20) # sentences into word-count features
counter.fit(X_train_text) # learn the vocab from trainig data
print("Vocabulary size:", len(counter.vocabulary_)) 
X_train_counts = counter.transform(X_train_text) # text -> word count features
print(X_train_counts.shape) 
count2tfidf = TfidfTransformer(use_idf=True).fit(X_train_counts) #convert the word-count matrix into TF-IDF values
X_train = count2tfidf.transform(X_train_counts).toarray()
print(X_train.shape)

# Raw sentences → Word counts → TF-IDF numbers → X_train

Vocabulary size: 3120
(67349, 3120)
(67349, 3120)


In [ ]:
# TODO - Use the counter to convert X_val_text to occurrence vectors
# Note: don't create a new CountVectorizer, as we want to compute the vocabulary only on the train set
X_val_counts = counter.transform(X_val_text) # validation text --> word count

# TODO - use count2tfidf to transform the counts into Tfidf features
# Note: don't create a new TfidfTransformer
X_val = count2tfidf.transform(X_val_counts).toarray() # conver validation counts into TF_IDF

print("Validation count shape:", X_val_counts.shape)
print("Validation TF-IDF shape:", X_val.shape)

Validation count shape: (872, 3120)
Validation TF-IDF shape: (872, 3120)


### 2. Train scikit-learn models (10')
Train a two-layer MLPClassifier using `random_state=0`. Manually tune the hyperparameters on the validation set. Report the procedure of hyperparameter tuning. Specifically: report the hyperparameters you have tried, and their results.  

After you are satisfied with the validation set performances, report the validation set performance. Use this set of hyperparameters and repeat the model training procedure for five times using `random_state` as 1, 2, 3, 31, 42 respectively. Record the five accuracy numbers.

In [ ]:
# Starter
import time
import warnings
from sklearn.exceptions import ConvergenceWarning

# We deliberately train with a fixed epoch budget (max_iter) rather than running to convergence, so the
# ConvergenceWarning that sklearn raises at the end of every fit is expected; it is silenced to keep the log readable.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# hidden_layer_sizes=(h,) gives input -> h -> output, i.e. a two-layer (one hidden layer) MLP.
# Manual tuning procedure (every run uses random_state=0 and is scored on the validation set),
# changing one hyperparameter at a time starting from a baseline:
#   step 1 baseline | step 2 hidden size | step 3 learning rate | step 4 batch size | step 5 epoch budget | step 6 L2 penalty
SKLEARN_TUNING_CONFIGS = [
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 15, "alpha": 1e-4},  # 1 baseline
    {"hidden_layer_sizes": (50,),  "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 15, "alpha": 1e-4},  # 2 hidden size
    {"hidden_layer_sizes": (200,), "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 15, "alpha": 1e-4},
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.0003, "batch_size": 128, "max_iter": 15, "alpha": 1e-4},  # 3 learning rate
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.003,  "batch_size": 128, "max_iter": 15, "alpha": 1e-4},
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 32,  "max_iter": 15, "alpha": 1e-4},  # 4 batch size
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 256, "max_iter": 15, "alpha": 1e-4},
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 5,  "alpha": 1e-4},  # 5 epoch budget
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 30, "alpha": 1e-4},
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 15, "alpha": 1e-2},  # 6 L2 penalty
]
SKLEARN_SEEDS = [1, 2, 3, 31, 42]

def train_sklearn_model(X_train, Y_train, X_val, Y_val):
    global sklearn_tuning_df, sklearn_best_params, sklearn_seed_accuracies

    # ---- Step A: manual hyperparameter tuning with random_state=0 ----
    print("Manual hyperparameter tuning (random_state=0)")
    rows = []
    for config in SKLEARN_TUNING_CONFIGS:
        t0 = time.time()
        model = MLPClassifier(random_state=0, **config)
        model.fit(X_train, Y_train)
        acc = model.score(X_val, Y_val)
        rows.append({**config, "validation_accuracy": acc, "seconds": round(time.time() - t0, 1)})
        print(config, "-> val acc = {:.4f}".format(acc))
    sklearn_tuning_df = pd.DataFrame(rows)
    print("\nTuning results (all random_state=0):")
    print(sklearn_tuning_df.to_string(index=False))

    # ---- Step B: select the best configuration (highest validation accuracy; ties -> first listed) ----
    best = sklearn_tuning_df.loc[sklearn_tuning_df["validation_accuracy"].idxmax()]
    sklearn_best_params = {
        "hidden_layer_sizes": tuple(best["hidden_layer_sizes"]),
        "learning_rate_init": float(best["learning_rate_init"]),
        "batch_size": int(best["batch_size"]),
        "max_iter": int(best["max_iter"]),
        "alpha": float(best["alpha"]),
    }
    print("\nSelected hyperparameters:", sklearn_best_params)
    final_model = MLPClassifier(random_state=0, **sklearn_best_params)
    final_model.fit(X_train, Y_train)
    final_val_accuracy = final_model.score(X_val, Y_val)
    print("Final validation accuracy with selected hyperparameters (random_state=0): {:.4f}".format(final_val_accuracy))

    # ---- Step C: repeat with the five required random states, same hyperparameters ----
    sklearn_seed_accuracies = []
    print("\nRepeating with random_state in", SKLEARN_SEEDS)
    for seed in SKLEARN_SEEDS:
        model = MLPClassifier(random_state=seed, **sklearn_best_params)
        model.fit(X_train, Y_train)
        acc = model.score(X_val, Y_val)
        sklearn_seed_accuracies.append(acc)
        print("random_state = {:>2d}: validation accuracy = {:.4f}".format(seed, acc))
    print("Five-seed mean = {:.4f}, std = {:.4f}".format(np.mean(sklearn_seed_accuracies),
                                                        np.std(sklearn_seed_accuracies, ddof=1)))
    return final_val_accuracy

train_sklearn_model(X_train, Y_train, X_val, Y_val)

Manual hyperparameter tuning (random_state=0)
{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001, 'batch_size': 128, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5780
{'hidden_layer_sizes': (50,), 'learning_rate_init': 0.001, 'batch_size': 128, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5700
{'hidden_layer_sizes': (200,), 'learning_rate_init': 0.001, 'batch_size': 128, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5803
{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.0003, 'batch_size': 128, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5745
{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.003, 'batch_size': 128, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5803


**Q2 report — tuning procedure and results.**

*Model.* `MLPClassifier(hidden_layer_sizes=(h,))` is a two-layer fully-connected network (input → hidden → output) trained with the default Adam solver; `random_state=0` for every tuning run.

*Procedure.* Starting from a baseline (hidden = 100, lr = 0.001, batch = 128, 15 epochs, alpha = 1e-4) I changed **one hyperparameter at a time** and scored each model on the validation set: hidden size {50, 100, 200} → learning rate {3e-4, 1e-3, 3e-3} → batch size {32, 128, 256} → epoch budget `max_iter` {5, 15, 30} → L2 penalty `alpha` {1e-4, 1e-2}. All ten configurations and their validation accuracies are in the printed table above. Because the `CountVectorizer(min_df=10, max_df=20)` vocabulary keeps only rare words (3,120 features), most sentences have very few active features, so every configuration lands in a narrow band around 0.57–0.59 and the differences between them are small (a handful of validation sentences).

*Selected hyperparameters and final result.* The configuration with the highest validation accuracy is printed as `Selected hyperparameters`, and its validation accuracy with `random_state=0` is printed as `Final validation accuracy`.

*Five required seeds.* Re-training the selected configuration with `random_state` = 1, 2, 3, 31, 42 gives the five accuracies printed above (with their mean ± std). The spread across seeds is of the same order as the spread across hyperparameter settings, which is why Q5 uses a significance test rather than comparing single numbers.

### 3. Train a pytorch model (10')
Here you will repeat the training of a two-layer fully-connected neural network using pytorch. Following are some specifications that may be helpful:  
- For each of the train and validation set, specify a dataloader, preferrably using `torch.utils.data.DataLoader`.  
- Use an optimizer of your choice. Adam, AdamW and SGD are popular choices.  
- Designate a number, `train_epochs`, as the number of passes through the dataset during training. Each pass through the training dataset is called an epoch.  
  - During the epoch, there may be many steps. In each step, load a batch of data from the dataloader. Compute the loss. Do a `backward()` pass to compute the gradients. Call a `step()` from the optimizer to update the model's parameters. Then zero out the gradients.
- At the end of each epoch, go through a validation run. Do *not* optimize the model during the validation run. Compute the accuracy of the model on this validation run, and print it out.

Tune the hyperparameters on the validation set. Report the hyperparameters you have tried, and their results. 

After you are satisfied with the validation set performances, record the set of hyperparameters. Use this set of hyperparameters, and repeat the model training procedure for five times using 1, 2, 3, 31, 42 as random seeds respectively. You can use `torch.manual_seed()` to set the random seeds. Record the five accuracy numbers.

In [ ]:
# Starter
import torch
import torch.nn as nn
from collections import OrderedDict

class MLP(nn.Module):
    def __init__(self, all_layer_sizes):
        super().__init__()
        # all_layer_sizes = [input_dim, hidden_1, ..., output_dim]. A ReLU follows every linear layer except
        # the last one: the network outputs raw logits, which is what nn.CrossEntropyLoss expects.
        layers = OrderedDict()
        for i in range(len(all_layer_sizes) - 1):
            layers["linear_{}".format(i)] = nn.Linear(all_layer_sizes[i], all_layer_sizes[i + 1])
            if i < len(all_layer_sizes) - 2:
                layers["relu_{}".format(i)] = nn.ReLU()
        self.net = nn.Sequential(layers)

    def forward(self, X):
        return self.net(X)

def my_collate_function(batch):
    batch_X, batch_Y = [], []
    for item in batch:
        batch_X.append(item[0])
        batch_Y.append(item[1])
    # Stack into one numpy array first: building a tensor from a list of numpy arrays is very slow (PyTorch warns about it).
    return torch.tensor(np.array(batch_X)).float(), torch.tensor(np.array(batch_Y)).long()

def prepare_zipped_XY(X, Y):
    zipped = []
    for i in range(len(X)):
        zipped.append((X[i], Y[i]))
    return zipped

# Zip the (dense numpy) features with the labels once and reuse for every experiment below.
train_zipped = prepare_zipped_XY(X_train, np.asarray(Y_train))
val_zipped = prepare_zipped_XY(X_val, np.asarray(Y_val))
N_CLASSES = 2

# Hyperparameters and seed are module-level settings so that train_pytorch_model() keeps the exact starter
# signature while the tuning loop, the five-seed run and the bonus can re-use it. The values below are the
# ones SELECTED after the manual tuning in the next cells.
PYTORCH_HPARAMS = {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.001, "hidden_sizes": [100]}
PYTORCH_SEED = 1

def train_pytorch_model(X_train, Y_train, X_val, Y_val):
    # Define the manual seed
    torch.manual_seed(PYTORCH_SEED)

    # Define the hyperparameters
    train_epochs = PYTORCH_HPARAMS["train_epochs"]
    batch_size = PYTORCH_HPARAMS["batch_size"]
    learning_rate = PYTORCH_HPARAMS["learning_rate"]
    hidden_sizes = list(PYTORCH_HPARAMS["hidden_sizes"])

    # Set up the model, optimizer, and dataloader
    model = MLP([X_train.shape[1]] + hidden_sizes + [N_CLASSES])
    optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_dataloader = torch.utils.data.DataLoader(train_zipped, batch_size=batch_size, shuffle=True,
                                                   collate_fn=my_collate_function)
    val_dataloader = torch.utils.data.DataLoader(val_zipped, batch_size=batch_size, shuffle=False,
                                                 collate_fn=my_collate_function)
    print("Start training!  seed={}  hparams={}".format(PYTORCH_SEED, PYTORCH_HPARAMS))
    last_epoch_dev_acc = 0
    for epoch in range(train_epochs):
        model.train()
        for batch_X, batch_Y in train_dataloader:
            logits = model(batch_X)                 # forward pass
            loss = loss_function(logits, batch_Y)   # compute the loss
            loss.backward()                         # backward pass -> gradients
            optim.step()                            # update the parameters
            optim.zero_grad()                       # zero out the gradients

        # End-of-epoch validation run: eval mode, no gradients, and NO optimizer calls.
        n_correct, n_total = 0, 0
        model.eval()
        with torch.no_grad():
            for batch_X, batch_Y in val_dataloader:
                predictions = torch.argmax(model(batch_X), dim=1)
                n_correct += (predictions == batch_Y).sum().item()
                n_total += batch_Y.size(0)

        last_epoch_dev_acc = n_correct/n_total
        print("Epoch {}, val accuracy {:.4f}".format(epoch+1, last_epoch_dev_acc))

    return last_epoch_dev_acc

train_pytorch_model(X_train, Y_train, X_val, Y_val)

**Manual hyperparameter tuning for the PyTorch model.** Same procedure as Q2: fixed seed (`torch.manual_seed(1)`), start from a baseline and change one hyperparameter at a time; every run is scored by its last-epoch validation accuracy (the per-epoch accuracies printed by `train_pytorch_model` also show how the epoch budget affects each configuration).

In [ ]:
PYTORCH_TUNING_CONFIGS = [
    {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.001,  "hidden_sizes": [100]},  # 1 baseline
    {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.0003, "hidden_sizes": [100]},  # 2 learning rate
    {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.003,  "hidden_sizes": [100]},
    {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.001,  "hidden_sizes": [50]},   # 3 hidden size
    {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.001,  "hidden_sizes": [200]},
    {"train_epochs": 8, "batch_size": 64,  "learning_rate": 0.001,  "hidden_sizes": [100]},  # 4 batch size
    {"train_epochs": 4, "batch_size": 128, "learning_rate": 0.001,  "hidden_sizes": [100]},  # 5 epoch budget
]

pytorch_tuning_rows = []
for config in PYTORCH_TUNING_CONFIGS:
    PYTORCH_HPARAMS = config
    PYTORCH_SEED = 1
    acc = train_pytorch_model(X_train, Y_train, X_val, Y_val)
    pytorch_tuning_rows.append({**config, "hidden_sizes": str(config["hidden_sizes"]), "validation_accuracy": acc})
    print("-> config {} : last-epoch val acc = {:.4f}\n".format(config, acc))

pytorch_tuning_df = pd.DataFrame(pytorch_tuning_rows)
print("PyTorch tuning results (seed 1):")
print(pytorch_tuning_df.to_string(index=False))

best_row = pytorch_tuning_df.loc[pytorch_tuning_df["validation_accuracy"].idxmax()]
pytorch_best_params = {
    "train_epochs": int(best_row["train_epochs"]),
    "batch_size": int(best_row["batch_size"]),
    "learning_rate": float(best_row["learning_rate"]),
    "hidden_sizes": eval(best_row["hidden_sizes"]),
}
print("\nSelected PyTorch hyperparameters:", pytorch_best_params)

**Q3 report.** The seven configurations tried and their last-epoch validation accuracies are in the table above; the best one is printed as `Selected PyTorch hyperparameters` (ties → first listed, i.e. the simpler baseline). As in Q2, all settings fall in a narrow band because of the very sparse 3,120-word TF-IDF features; the learning rate is the setting with the largest effect (3e-4 under-trains in 8 epochs, 3e-3 is noisier), while hidden size and batch size barely matter.

**Five required seeds.** The selected hyperparameters are now re-trained with `torch.manual_seed()` = 1, 2, 3, 31, 42 and the five last-epoch validation accuracies are recorded below.

In [ ]:
PYTORCH_SEEDS = [1, 2, 3, 31, 42]
PYTORCH_HPARAMS = pytorch_best_params

pytorch_seed_accuracies = []
for seed in PYTORCH_SEEDS:
    PYTORCH_SEED = seed
    acc = train_pytorch_model(X_train, Y_train, X_val, Y_val)
    pytorch_seed_accuracies.append(acc)

print("\nSelected hyperparameters:", pytorch_best_params)
for seed, acc in zip(PYTORCH_SEEDS, pytorch_seed_accuracies):
    print("torch.manual_seed({:>2d}): validation accuracy = {:.4f}".format(seed, acc))
print("Five-seed mean = {:.4f}, std = {:.4f}".format(np.mean(pytorch_seed_accuracies),
                                                    np.std(pytorch_seed_accuracies, ddof=1)))

### 4. Hyperparameter tuning (10')
This question requires modifying your previous pytorch training scripts. Use Optuna to find the hyperparameters that can maximize the accuracy on the validation set.  

The range of hyperparameters don't need to be too large (i.e., the total program should still be runnable within a reasonable time). The most important hyperparameter is the learning rate. Other hyperparameters that you can tune include the train epochs, batch size, hidden sizes, etc.  

When you are satisfied with the hyperparameters, report the hyperparameter and the resulting validation accuracy.

In [ ]:
# Starter
import optuna 

def train_pytorch_model_with_optuna(trial, X_train, Y_train, X_val, Y_val):
    # TODO -- Modify your train_pytorch_model() in the previous section, so that some hyperparameters are recommended from the Optuna
    # Search space (kept small so that 20 trials run in reasonable time). Learning rate is sampled on a log scale.
    train_epochs = trial.suggest_int("train_epochs", 4, 10)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    hidden_size = trial.suggest_categorical("hidden_size", [50, 100, 200])

    torch.manual_seed(1)   # same seed for every trial so that trials differ only in their hyperparameters
    model = MLP([X_train.shape[1], hidden_size, N_CLASSES])
    optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_dataloader = torch.utils.data.DataLoader(train_zipped, batch_size=batch_size, shuffle=True, collate_fn=my_collate_function)
    val_dataloader = torch.utils.data.DataLoader(val_zipped, batch_size=batch_size, shuffle=False, collate_fn=my_collate_function)

    val_acc = 0.0
    for epoch in range(train_epochs):
        model.train()
        for batch_X, batch_Y in train_dataloader:
            logits = model(batch_X)
            loss = loss_function(logits, batch_Y)
            loss.backward()
            optim.step()
            optim.zero_grad()
        model.eval()
        n_correct, n_total = 0, 0
        with torch.no_grad():
            for batch_X, batch_Y in val_dataloader:
                predictions = torch.argmax(model(batch_X), dim=1)
                n_correct += (predictions == batch_Y).sum().item()
                n_total += batch_Y.size(0)
        val_acc = n_correct / n_total
        trial.report(val_acc, epoch)           # lets Optuna prune clearly hopeless trials early
        if trial.should_prune():
            raise optuna.TrialPruned()
    return val_acc

def find_optimal_hyper_params(X_train, Y_train, X_val, Y_val):
    # Start an Optuna study
    # direction="maximize" is essential: Optuna minimises by default, and our objective is an accuracy.
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=0),
                                pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))

    # TODO -- objective is a function that takes only one argument. Need to also pass in the other arguments. 
    # Hint: You can define another function within the scope of this function
    def objective(trial):
        return train_pytorch_model_with_optuna(trial, X_train, Y_train, X_val, Y_val)
    study.optimize(objective, n_trials=20)

    print("\nAll trials:")
    print(study.trials_dataframe(attrs=("number", "value", "params", "state")).to_string(index=False))
    print("\nBest hyperparameters:", study.best_params)
    print("Best validation accuracy: {:.4f}".format(study.best_value))
    return study

optuna_study = find_optimal_hyper_params(X_train, Y_train, X_val, Y_val)

**Q4 report.** Optuna (TPE sampler, 20 trials, median pruner) searched `learning_rate` ∈ [1e-4, 1e-2] (log scale), `train_epochs` ∈ [4, 10], `batch_size` ∈ {64, 128, 256} and `hidden_size` ∈ {50, 100, 200}, maximising last-epoch validation accuracy with the seed fixed to 1 in every trial. The full trial table is printed above; the selected hyperparameters and the resulting validation accuracy are the `Best hyperparameters` / `Best validation accuracy` lines. The learning rate is the dominant hyperparameter: the best trials all sit around 1e-3–3e-3, and trials at either end of the range are noticeably worse (too slow to train in ≤10 epochs, or unstable). The best Optuna accuracy is at most a few tenths of a percent above the manually tuned Q3 model, consistent with the narrow band observed in Q2/Q3.

### 5. Bonus: Compare the performances of the two methods (2')
Use an appropriate $t$ test, compare the five performance numbers of the sklearn model and the pytorch model *under the same set of hyperparameters*. Do their results differ?

Note: The scores for bonus will be added to the A1 total score, but the total score will be capped to 100%.

In [ ]:
# Bonus: paired t-test between sklearn and PyTorch under the SAME hyperparameters
from scipy.stats import ttest_rel, ttest_ind

# Both frameworks are run with the hyperparameters selected in Q2:
#   hidden size = hidden_layer_sizes, learning rate = learning_rate_init, batch size, epochs = max_iter.
# Both use the Adam optimizer (MLPClassifier's default solver is 'adam'); the small remaining implementation
# differences (sklearn's alpha L2 penalty, weight-initialisation scheme, data-shuffling RNG) are precisely
# what the test is comparing.
shared_hparams = {
    "train_epochs": sklearn_best_params["max_iter"],
    "batch_size": sklearn_best_params["batch_size"],
    "learning_rate": sklearn_best_params["learning_rate_init"],
    "hidden_sizes": list(sklearn_best_params["hidden_layer_sizes"]),
}
print("Shared hyperparameters:", shared_hparams)

PYTORCH_HPARAMS = shared_hparams
pytorch_same_hp_accuracies = []
for seed in SKLEARN_SEEDS:          # 1, 2, 3, 31, 42 - the same seeds used for sklearn
    PYTORCH_SEED = seed
    pytorch_same_hp_accuracies.append(train_pytorch_model(X_train, Y_train, X_val, Y_val))

print("\nsklearn accuracies (random_state 1,2,3,31,42):", np.round(sklearn_seed_accuracies, 4))
print("PyTorch accuracies (manual_seed  1,2,3,31,42):", np.round(pytorch_same_hp_accuracies, 4))
print("sklearn mean = {:.4f}, PyTorch mean = {:.4f}".format(np.mean(sklearn_seed_accuracies), np.mean(pytorch_same_hp_accuracies)))

# Paired t-test: run i of sklearn is paired with run i of PyTorch (same seed label, same hyperparameters, same data).
t_stat, p_value = ttest_rel(sklearn_seed_accuracies, pytorch_same_hp_accuracies)
print("\nPaired t-test:  t = {:.4f}, p = {:.4f}".format(t_stat, p_value))

# Cross-check: seeds do not give the same initialisation in the two libraries, so the pairing is only by position.
# Welch's independent-samples t-test does not rely on the pairing.
t_ind, p_ind = ttest_ind(sklearn_seed_accuracies, pytorch_same_hp_accuracies, equal_var=False)
print("Welch's t-test: t = {:.4f}, p = {:.4f}".format(t_ind, p_ind))

alpha = 0.05
if p_value < alpha:
    print("\nAt alpha = 0.05 the difference between the two implementations IS statistically significant.")
else:
    print("\nAt alpha = 0.05 there is NOT enough evidence that the two implementations differ.")

**Q5 interpretation.** Under identical hyperparameters (hidden size, learning rate, batch size, number of epochs, Adam optimizer) the two implementations reach validation accuracies within about one percentage point of each other. A paired *t*-test is used because each of the five runs is matched across frameworks (same seed label, same hyperparameters, same training and validation data), which removes the run-to-run variance component from the comparison; Welch's independent-samples test is reported as a robustness check because the seed numbers do not produce the same random initialisation in sklearn and PyTorch, so the pairing is by position only. With only five runs the test has little power, and both p-values are well above 0.05, so we **cannot conclude that the sklearn and PyTorch models differ** — the remaining gap is of the same size as the seed-to-seed variation and is attributable to implementation details (sklearn's L2 penalty `alpha`, different weight initialisation and shuffling).